In [73]:
import json

with open("dw_data/document_corpus_dw.json", "r", encoding="utf-8") as f:
    document_corpus = json.load(f)

with open("dw_data/first_example_10_queries.json", "r", encoding="utf-8") as f:
    my_queries = json.load(f)

queries = my_queries["queries"]
answers = my_queries["answers"]

In [74]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import umap

from sentence_transformers import SentenceTransformer, util

In [75]:
model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4359.24it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [76]:
texts = []
ids = []

for item in document_corpus.values():
    ids.append(item["id"])
    texts.append((item["title"] + " " + item["description"]).lower().strip())

In [77]:
embeddings = model.encode(texts, convert_to_numpy=True)

In [78]:
for i, query in enumerate(queries):
    query_embedding = model.encode(query)
    cosine_scores = util.cos_sim(query_embedding, embeddings)[0]

    top_k = min(5, len(cosine_scores))

    relevant_docs = answers[i]
    correct_count = 0

    print(f"\nQuery: {query}")

    for idx in top_results:
        doc_id = ids[idx]
        print(f"Document ID: {doc_id}, Score: {cosine_scores[idx]:.4f}")
        if doc_id in relevant_docs:
            correct_count += 1

    print(f"{correct_count}/{top_k} documents were a right guess")


Query: Doctor fights with Weeping Angels and wants to save Amy
Document ID: 1x8, Score: 0.2503
Document ID: 1x1, Score: 0.3040
Document ID: 2x1, Score: 0.3364
Document ID: 1x4, Score: 0.1469
Document ID: 2x2, Score: 0.2745
0/5 documents were a right guess

Query: Doctor and Clara are in nineteenth century
Document ID: 1x8, Score: 0.1212
Document ID: 1x1, Score: 0.2377
Document ID: 2x1, Score: 0.2663
Document ID: 1x4, Score: 0.3414
Document ID: 2x2, Score: 0.3823
0/5 documents were a right guess

Query: Doctor meet River Song for the first time
Document ID: 1x8, Score: 0.1035
Document ID: 1x1, Score: 0.3305
Document ID: 2x1, Score: 0.3442
Document ID: 1x4, Score: 0.2406
Document ID: 2x2, Score: 0.3050
0/5 documents were a right guess

Query: Doctor and Donna encounter Daleks and Davros
Document ID: 1x8, Score: 0.1912
Document ID: 1x1, Score: 0.3203
Document ID: 2x1, Score: 0.3628
Document ID: 1x4, Score: 0.1887
Document ID: 2x2, Score: 0.3217
0/5 documents were a right guess

Query: Do

In [79]:
model_more_precise = SentenceTransformer('all-mpnet-base-v2')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7728.47it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [80]:
embeddings_precised = model_more_precise.encode(texts, convert_to_numpy=True)

In [81]:
for i, query in enumerate(queries):
    query_embedding = model_more_precise.encode(query)
    cosine_scores = util.cos_sim(query_embedding, embeddings_precised)[0]

    top_k = min(5, len(cosine_scores))

    relevant_docs = answers[i]
    correct_count = 0

    print(f"\nQuery: {query}")

    for idx in top_results:
        doc_id = ids[idx]
        print(f"Document ID: {doc_id}, Score: {cosine_scores[idx]:.4f}")
        if doc_id in relevant_docs:
            correct_count += 1

    print(f"{correct_count}/{top_k} documents were a right guess")


Query: Doctor fights with Weeping Angels and wants to save Amy
Document ID: 1x8, Score: 0.3547
Document ID: 1x1, Score: 0.5546
Document ID: 2x1, Score: 0.4744
Document ID: 1x4, Score: 0.3305
Document ID: 2x2, Score: 0.3295
0/5 documents were a right guess

Query: Doctor and Clara are in nineteenth century
Document ID: 1x8, Score: 0.1829
Document ID: 1x1, Score: 0.3686
Document ID: 2x1, Score: 0.3828
Document ID: 1x4, Score: 0.4261
Document ID: 2x2, Score: 0.3667
0/5 documents were a right guess

Query: Doctor meet River Song for the first time
Document ID: 1x8, Score: 0.2910
Document ID: 1x1, Score: 0.5287
Document ID: 2x1, Score: 0.4118
Document ID: 1x4, Score: 0.3286
Document ID: 2x2, Score: 0.3014
0/5 documents were a right guess

Query: Doctor and Donna encounter Daleks and Davros
Document ID: 1x8, Score: 0.1384
Document ID: 1x1, Score: 0.3686
Document ID: 2x1, Score: 0.3328
Document ID: 1x4, Score: 0.3552
Document ID: 2x2, Score: 0.2341
0/5 documents were a right guess

Query: Do

In [82]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer, util
import numpy as np

document_list = [v for k, v in document_corpus.items()]

# Теперь можно строить тексты и ids
texts_normalized = [(f"{doc['id']} {doc['title']} {doc['description']}").lower().strip() 
                    for doc in document_list]
ids = [doc['id'] for doc in document_list]

# TF-IDF векторизация
tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(texts_normalized)

model_for_tfidf = SentenceTransformer('all-mpnet-base-v2')
embeddings_for_tfidf = model_for_tfidf.encode(texts_normalized, convert_to_tensor=True)

st_correct_list = []

# Пример цикла для поиска
for i, query in enumerate(queries):
    query_norm = query.lower().strip()

    # TF-IDF similarity
    query_vec_tfidf = tfidf.transform([query_norm])
    cosine_tfidf = (tfidf_matrix @ query_vec_tfidf.T).toarray().ravel()

    # Sentence Transformers similarity
    query_emb = model_more_precise.encode(query_norm, convert_to_tensor=True)
    cosine_st = util.cos_sim(query_emb, embeddings_for_tfidf)[0].cpu().numpy()

    # Комбинируем (вес можно подбирать)
    combined_scores = 0.5 * cosine_tfidf + 0.5 * cosine_st

    # Top-5
    top_k = min(5, len(combined_scores))
    top_results = np.argsort(-combined_scores)[:top_k]

    relevant_docs = answers[i]
    correct_count = 0

    print(f"\nQuery: {query}")
    print(f"Relevant answers: {relevant_docs}")

    for idx in top_results:
        doc_id = ids[idx]
        print(f"Document ID: {doc_id}, Score: {combined_scores[idx]:.4f}")
        if doc_id in relevant_docs:
            correct_count += 1

    print(f"{correct_count}/{top_k} documents were a right guess")
    st_correct_list.append(correct_count)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2360.68it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Query: Doctor fights with Weeping Angels and wants to save Amy
Relevant answers: ['5x4', '5x5', '7x5', '6x11', '3x10']
Document ID: 7x5, Score: 0.5400
Document ID: 5x4, Score: 0.5139
Document ID: 5x5, Score: 0.3897
Document ID: 6x11, Score: 0.3476
Document ID: 5x1, Score: 0.3267
4/5 documents were a right guess

Query: Doctor and Clara are in nineteenth century
Relevant answers: ['7x6', '7x12', '7x8', '1x3', '7x10']
Document ID: 7x12, Score: 0.3676
Document ID: 7x14, Score: 0.3503
Document ID: 9x1, Score: 0.3351
Document ID: 2x4, Score: 0.3348
Document ID: 8x4, Score: 0.3347
1/5 documents were a right guess

Query: Doctor meet River Song for the first time
Relevant answers: ['4x8', '4x9', '5x4', '5x5', '6x1']
Document ID: 6x13, Score: 0.4282
Document ID: 9x13, Score: 0.3921
Document ID: 5x4, Score: 0.3907
Document ID: 6x1, Score: 0.3705
Document ID: 7x5, Score: 0.3125
2/5 documents were a right guess

Query: Doctor and Donna encounter Daleks and Davros
Relevant answers: ['4x12', '4x13

In [83]:
df = pd.read_csv("dw_data/search_results_summary.csv")
df["ST Correct"] = st_correct_list

# Сохраняем обратно в CSV
df.to_csv("dw_data/search_results_summary.csv", index=False)

print(df)

                                               Query  Boolean Correct  \
0  Doctor fights with Weeping Angels and wants to...                0   
1         Doctor and Clara are in nineteenth century                1   
2          Doctor meet River Song for the first time                0   
3       Doctor and Donna encounter Daleks and Davros                0   
4      Doctor and Rose end up in a parallel universe                0   
5                              Doctor meets van Gogh                0   
6  Doctor and Martha face time paradox creatures ...                0   
7                 Doctor and Bill encounter Cybermen                0   
8   Paternoster Gang and Doctor and Vastra and Strax                1   
9         Doctor and Rose meet her father Pete Tyler                0   

   BM25 Correct  ST Correct  
0             4           4  
1             1           1  
2             2           2  
3             2           1  
4             1           1  
5             1 